# Deliver Q4 2025's channel report in Excel

*Data Analysis in Python · Topic 4.6*

The CMO wants the fourth quarter of 2025's channel report as an Excel file, formatted so it can go straight into a board deck.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds `sessions` (lesson 3.'s steps) and the cell that adds its `channel_group` column (topic 4.2.'s steps).
2. In the first cell marked ✏️, create:
   - `by_channel`: one row per channel group, with the channel groups as the index, and the columns `sessions`, `transactions`, `revenue` and `conversion_rate`, sorted by revenue, highest first (topic 4.3.)
   - `monthly_rates`: months as rows (`"2025-10"` to `"2025-12"`) and channel groups as columns, filled with each month and channel's conversion rate, from its totals (topic 4.5.)
3. In the second cell marked ✏️, write `format_column(sheet, header, number_format)`, which gives every cell under a header the number format.
4. In the last cell marked ✏️, write `REPORT_FILE` with two sheets:
   - **Channels**: `by_channel`, with `sessions` and `transactions` as `#,##0`, `revenue` as `$#,##0.00` and `conversion_rate` as `0.00%`
   - **Conversion by month**: `monthly_rates`, with every rate as `0.00%`

   Make every column of both sheets 16 wide. Then read the file back and compare it with your DataFrames.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. Then download the file from the Files panel and open it.

**Expected output**: `q4_2025_channels.xlsx` has two sheets. **Channels** has nine channel groups, Organic Search first with $302,904.90, with sessions and transactions as `#,##0`, revenue as `$#,##0.00` and conversion rates as `0.00%`: Email's is 7.95%. **Conversion by month** has three months and nine channel groups, every rate as `0.00%`: Email's December is 9.22%. Every column is 16 wide.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-10", "2025-11", "2025-12"]
START, END = "2025-10-01", "2025-12-31"
REPORT_FILE = "q4_2025_channels.xlsx"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

## The two tables

In [ ]:
by_channel = sessions.groupby("channel_group").agg({"sessions": "sum", "transactions": "sum", "revenue": "sum"})
by_channel["conversion_rate"] = by_channel["transactions"] / by_channel["sessions"]
by_channel = by_channel.sort_values("revenue", ascending=False)

sessions["month"] = sessions["date"].dt.strftime("%Y-%m")
monthly = sessions.groupby(["month", "channel_group"], as_index=False)[["sessions", "transactions"]].sum()
monthly["conversion_rate"] = monthly["transactions"] / monthly["sessions"]
monthly_rates = monthly.pivot(index="month", columns="channel_group", values="conversion_rate")
by_channel, monthly_rates

## The Excel file

In [ ]:
def format_column(sheet, header, number_format):
    """Give every cell under the header the number format."""
    for cell in sheet[1]:
        if cell.value == header:
            for value_cell in sheet[cell.column_letter][1:]:
                value_cell.number_format = number_format

In [ ]:
with pd.ExcelWriter(REPORT_FILE, engine="openpyxl") as writer:
    by_channel.to_excel(writer, sheet_name="Channels")
    monthly_rates.to_excel(writer, sheet_name="Conversion by month")

    channels = writer.sheets["Channels"]
    format_column(channels, "sessions", "#,##0")
    format_column(channels, "transactions", "#,##0")
    format_column(channels, "revenue", "$#,##0.00")
    format_column(channels, "conversion_rate", "0.00%")

    rates = writer.sheets["Conversion by month"]
    for group in monthly_rates.columns:
        format_column(rates, group, "0.00%")

    for sheet in [channels, rates]:
        for column in sheet.columns:
            sheet.column_dimensions[column[0].column_letter].width = 16

back = pd.read_excel(REPORT_FILE, sheet_name=None, index_col=0)
list(back), (back["Channels"]["revenue"] - by_channel["revenue"]).abs().max(), (back["Conversion by month"] - monthly_rates).abs().max().max()

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, the cells that have run and the file you wrote. It doesn't change anything.
def check_your_work():
    import os
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    sessions = found.get("sessions")
    if not isinstance(sessions, pd.DataFrame) or len(sessions) != 5895 or sessions["sessions"].sum() != 486702 or "channel_group" not in sessions.columns:
        results.append((False, "sessions should be the clean table for October to December 2025, with its channel_group column. Run the cells that build it, and don't change it."))
        expected = None
    else:
        expected = sessions.groupby("channel_group")[["sessions", "transactions", "revenue"]].sum()
        expected["conversion_rate"] = expected["transactions"] / expected["sessions"]
        months = sessions.assign(month=sessions["date"].dt.strftime("%Y-%m")).groupby(["month", "channel_group"])[["sessions", "transactions"]].sum()
        expected_rates = (months["transactions"] / months["sessions"]).reset_index(name="rate").pivot(index="month", columns="channel_group", values="rate")

    by_channel = found.get("by_channel")
    monthly_rates = found.get("monthly_rates")
    if expected is not None:
        if not isinstance(by_channel, pd.DataFrame) or len(by_channel) != 9 or "conversion_rate" not in by_channel.columns:
            results.append((False, "by_channel should have the nine channel groups as its index, with sessions, transactions, revenue and conversion_rate, as in topic 4.3."))
        elif not isinstance(monthly_rates, pd.DataFrame) or monthly_rates.shape != (3, 9):
            results.append((False, "monthly_rates should have the three months as rows and the nine channel groups as columns, filled with conversion rates."))
        elif abs(monthly_rates.loc["2025-12", "Email"] - expected_rates.loc["2025-12", "Email"]) > 1e-9:
            results.append((False, f"monthly_rates' December Email is {monthly_rates.loc['2025-12', 'Email']:.4f}, but its transactions over its sessions is {expected_rates.loc['2025-12', 'Email']:.4f}. Total each month and channel first, then divide."))
        else:
            results.append((True, "by_channel and monthly_rates have the quarter's channels, and the month's rates come from its totals."))

    report_file = found.get("REPORT_FILE")
    written = history is None or any(("ExcelWriter" in source or "to_excel" in source) and "def check_your_work" not in source for source in history)
    if not isinstance(report_file, str) or not os.path.exists(report_file) or not written:
        results.append((False, f"{report_file} hasn't been written in this session. Write it with pd.ExcelWriter(REPORT_FILE, engine=\"openpyxl\") and to_excel()."))
    elif expected is not None:
        import openpyxl
        back = pd.read_excel(report_file, sheet_name=None, index_col=0)
        book = openpyxl.load_workbook(report_file)
        if sorted(back) != ["Channels", "Conversion by month"]:
            results.append((False, f"{report_file} should have the sheets Channels and Conversion by month. It has {list(back)}."))
        else:
            channels = back["Channels"]
            if "Email" not in channels.index:
                results.append((False, "The Channels sheet has no channel names: write by_channel with its index (to_excel() without index=False)."))
            elif missing := [column for column in expected.columns if column not in channels.columns]:
                results.append((False, f"The Channels sheet has no {', '.join(missing)} column."))
            elif channels.loc["Email", "conversion_rate"] > 1:
                results.append((False, f"The Channels sheet's Email conversion rate is {channels.loc['Email', 'conversion_rate']:.2f}: it was multiplied by 100 before writing. Keep it a fraction, and let the 0.00% format show it as a percentage."))
            elif any((channels.loc[expected.index, column] - expected[column]).abs().max() > 0.005 for column in ["sessions", "transactions", "revenue"]) or (channels.loc[expected.index, "conversion_rate"] - expected["conversion_rate"]).abs().max() > 1e-9:
                results.append((False, "The Channels sheet's numbers don't match the quarter's totals per channel group. Write by_channel as it is."))
            elif list(channels.index) != list(expected.sort_values("revenue", ascending=False).index):
                results.append((False, "The Channels sheet isn't sorted by revenue, highest first: Organic Search should come first."))
            else:
                sheet = book["Channels"]
                letters = {cell.value: cell.column_letter for cell in sheet[1]}
                wanted = {"sessions": "#,##0", "transactions": "#,##0", "revenue": "$#,##0.00", "conversion_rate": "0.00%"}
                wrong = [f"{header} is {sheet[letters[header] + '2'].number_format}" for header, number_format in wanted.items() if sheet[letters[header] + "2"].number_format != number_format]
                if wrong:
                    results.append((False, f"Some Channels columns don't have their number format: {'; '.join(wrong)}. Use format_column(sheet, header, number_format) on writer.sheets[\"Channels\"]."))
                else:
                    results.append((True, "The Channels sheet has the nine groups, best revenue first, with numbers, dollars and percentages formatted."))

            rates = back["Conversion by month"]
            sheet = book["Conversion by month"]
            if rates.shape != (3, 9) or "2025-12" not in rates.index or "Email" not in rates.columns:
                results.append((False, f"The Conversion by month sheet should have the three months as rows and the nine channel groups as columns. It has {rates.shape[0]} rows and {rates.shape[1]} columns."))
            elif (rates.loc[expected_rates.index, expected_rates.columns] - expected_rates).abs().max().max() > 1e-9:
                results.append((False, "The Conversion by month sheet's rates don't match each month and channel's transactions over its sessions, as fractions."))
            elif any(cell.number_format != "0.00%" for row in sheet.iter_rows(min_row=2, min_col=2) for cell in row):
                results.append((False, "Some rates on the Conversion by month sheet aren't formatted as 0.00%. Format every cell below the header row and right of the months."))
            else:
                results.append((True, f"The Conversion by month sheet has three months of rates as percentages: Email's December is {expected_rates.loc['2025-12', 'Email']:.2%}."))

            narrow = [name for name in back if any((book[name].column_dimensions[column[0].column_letter].width or 0) < 16 for column in book[name].columns)]
            if narrow:
                results.append((False, f"Some columns on {', '.join(narrow)} are narrower than 16. Set sheet.column_dimensions[letter].width = 16 for every column."))
            else:
                results.append((True, "Every column on both sheets is 16 wide."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Download the file from the Files panel, and open it.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
by_channel = sessions.groupby("channel_group").agg({"sessions": "sum", "transactions": "sum", "revenue": "sum"})
by_channel["conversion_rate"] = by_channel["transactions"] / by_channel["sessions"]
by_channel = by_channel.sort_values("revenue", ascending=False)

sessions["month"] = sessions["date"].dt.strftime("%Y-%m")
monthly = sessions.groupby(["month", "channel_group"], as_index=False)[["sessions", "transactions"]].sum()
monthly["conversion_rate"] = monthly["transactions"] / monthly["sessions"]
monthly_rates = monthly.pivot(index="month", columns="channel_group", values="conversion_rate")
by_channel, monthly_rates
```

```python
def format_column(sheet, header, number_format):
    """Give every cell under the header the number format."""
    for cell in sheet[1]:
        if cell.value == header:
            for value_cell in sheet[cell.column_letter][1:]:
                value_cell.number_format = number_format
```

```python
with pd.ExcelWriter(REPORT_FILE, engine="openpyxl") as writer:
    by_channel.to_excel(writer, sheet_name="Channels")
    monthly_rates.to_excel(writer, sheet_name="Conversion by month")

    channels = writer.sheets["Channels"]
    format_column(channels, "sessions", "#,##0")
    format_column(channels, "transactions", "#,##0")
    format_column(channels, "revenue", "$#,##0.00")
    format_column(channels, "conversion_rate", "0.00%")

    rates = writer.sheets["Conversion by month"]
    for group in monthly_rates.columns:
        format_column(rates, group, "0.00%")

    for sheet in [channels, rates]:
        for column in sheet.columns:
            sheet.column_dimensions[column[0].column_letter].width = 16

back = pd.read_excel(REPORT_FILE, sheet_name=None, index_col=0)
list(back), (back["Channels"]["revenue"] - by_channel["revenue"]).abs().max(), (back["Conversion by month"] - monthly_rates).abs().max().max()
```

</details>